# PAF Experiments on Tabular Data

Pilot study extending *On Embeddings for Numerical Features in Tabular Deep Learning* (Gorishniy et al., NeurIPS 2022).

**Structure (mirrors original paper):**
1. Each model is first evaluated with **default HPs** (no tuning)
2. Then evaluated with **tuned HPs** (Optuna TPE, val R² as objective)
3. Final table shows both columns side by side — same format as Table 6 in the original paper

## Setup

In [11]:
!git clone https://github.com/ChistyakovArtem/advanced-tabular-embeddings /kaggle/working/paf_experiments

import sys
sys.path.insert(0, '/kaggle/working/paf_experiments/paf_experiments')

fatal: destination path '/kaggle/working/paf_experiments' already exists and is not an empty directory.


In [12]:
import os, sys, json

PROJECT_ROOT = '/kaggle/input/paf-experiments'
if os.path.exists(PROJECT_ROOT):
    sys.path.insert(0, PROJECT_ROOT)
else:
    sys.path.insert(0, '/kaggle/working')

DATA_ROOT    = '/kaggle/input/datasets/artemchistyakov/datasets-in-tabm/datasets_original'
RESULTS_DIR = '/kaggle/working/results'

import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PyTorch: 2.9.0+cu126
CUDA: True
GPU: Tesla T4


In [13]:
!pip install optuna -q

In [22]:
from data.loader          import load_dataset
from experiments.runner   import run_experiments, DefaultHParams, ALL_VARIANTS
from experiments.tuner    import tune_all, build_model_from_hp
from experiments.trainer  import train
from results.analysis     import aggregate, print_table

ALL_VARIANTS = [i for i in ALL_VARIANTS if 'PAFNet' not in i]

print('Variants:', len(ALL_VARIANTS))
for v in ALL_VARIANTS:
    print(' ', v)


Variants: 7
  MLP
  EmbMLP-orig-plain
  EmbMLP-orig-sc
  EmbMLP-orig-sc_af
  EmbMLP-grid-plain
  EmbMLP-grid-sc
  EmbMLP-grid-sc_af


## Config

In [27]:
DATASETS  = ['q_OnlineNewsPopularity', 'q_nyc-taxi-green-dec-2016']   # add more as needed: 'house', 'adult', etc.
N_SEEDS   = 3                # seeds for final evaluation
N_TRIALS  = 50               # Optuna trials per variant
N_EPOCHS  = 100
PATIENCE  = 16

# Default HPs (no tuning)
default_hp = DefaultHParams(
    hidden_dim=64, n_layers=2, k=16, dropout=0.1,
    lr=1e-3, weight_decay=1e-5,
    n_epochs=N_EPOCHS, patience=PATIENCE,
    batch_size=256, n_seeds=N_SEEDS,
)

## Load datasets

In [28]:
datasets = {}
for ds_name in DATASETS:
    datasets[ds_name] = load_dataset(ds_name, data_root=DATA_ROOT, batch_size=256)
    print(f"  {ds_name}: n_features={datasets[ds_name]['n_features']}")

  [q_OnlineNewsPopularity] n_features=59 (num=45, bin=14, cat_ohe=0, cat_num=0)
  q_OnlineNewsPopularity: n_features=59
  [q_nyc-taxi-green-dec-2016] n_features=25 (num=9, bin=3, cat_ohe=13, cat_num=0)
  q_nyc-taxi-green-dec-2016: n_features=25


## Part 1 — Default HPs

Run all variants with fixed hyperparameters, no tuning.
Quick baseline to see which directions are promising.

In [29]:
results_default = run_experiments(
    dataset_names=DATASETS,
    data_root=DATA_ROOT,
    results_dir=RESULTS_DIR + '/default',
    variants=ALL_VARIANTS,
    hp=default_hp,
    device=device,
    verbose=True,
    do_tune=False,
)


  Dataset: q_OnlineNewsPopularity
  [q_OnlineNewsPopularity] n_features=59 (num=45, bin=14, cat_ohe=0, cat_num=0)
  n_features=59  task=regression  n_classes=1

  [MLP]  seed=42
  [   1] train_loss=0.9458  val_loss=0.8903  val_R2=0.0859  *
  [  10] train_loss=0.8039  val_loss=0.8442  val_R2=0.1332
  [  20] train_loss=0.7668  val_loss=0.8550  val_R2=0.1221
  Early stop at epoch 27 (patience=16)
  → best_epoch=11  best_val_R2=0.1413  test_R2=0.1350  (6.4s)

  [MLP]  seed=43
  [   1] train_loss=0.9474  val_loss=0.8810  val_R2=0.0954  *
  [  10] train_loss=0.8018  val_loss=0.8393  val_R2=0.1382
  [  20] train_loss=0.7602  val_loss=0.8485  val_R2=0.1288
  Early stop at epoch 25 (patience=16)
  → best_epoch=9  best_val_R2=0.1424  test_R2=0.1363  (6.0s)

  [MLP]  seed=44
  [   1] train_loss=0.9639  val_loss=0.9068  val_R2=0.0689  *
  [  10] train_loss=0.8132  val_loss=0.8414  val_R2=0.1361  *
  [  20] train_loss=0.7617  val_loss=0.8434  val_R2=0.1341
  [  30] train_loss=0.7245  val_loss=0.86

## Part 2 — Tuned HPs (Optuna TPE)

For each variant: run N_TRIALS Optuna trials optimising val R²,
then re-evaluate best config with N_SEEDS seeds.

In [ ]:
import statistics, math
from experiments.tuner  import tune, build_model_from_hp
from experiments.runner import run_one
from pathlib import Path

results_tuned = []
tuning_info   = {}   # {ds_name: {model_name: tune_result}}

for ds_name in DATASETS:
    dataset = datasets[ds_name]
    tuning_info[ds_name] = {}
    print(f"\n{'='*60}\n  Tuning on: {ds_name}\n{'='*60}")

    for model_name in ALL_VARIANTS:
        if 'PAFNet-plain' in model_name:
            continue
        
        print(f"\n  [HPO] {model_name} — {N_TRIALS} trials...")
        t = tune(
            model_name=model_name,
            dataset=dataset,
            n_trials=N_TRIALS,
            device=device,
            seed=42,
            n_epochs=N_EPOCHS,
            patience=PATIENCE,
            show_progress=True,
        )
        tuning_info[ds_name][model_name] = t
        print(f"  [HPO] best_val_R2={t['best_val_r2']:.4f}  "
              f"hp={t['best_hp']}  lr={t['best_lr']:.2e}")

        # Evaluate best HP with N_SEEDS seeds
        seed_results = []
        for seed_idx in range(N_SEEDS):
            actual_seed = 42 + seed_idx
            print(f"  [{model_name}]  seed={actual_seed}  (tuned)")
            r = run_one(
                model_name=model_name,
                dataset=dataset,
                hp=default_hp,
                results_dir=Path(RESULTS_DIR + '/tuned'),
                device=device,
                seed=actual_seed,
                verbose=True,
                tuned_hp=t['best_hp'],
                tuned_lr=t['best_lr'],
                tuned_wd=t['best_wd'],
            )
            seed_results.append(r)
            results_tuned.append(r)

        vals  = [r['best_val_metric'] for r in seed_results]
        tests = [r['test_metric']     for r in seed_results]
        std_v = statistics.stdev(vals)  if len(vals)  > 1 else 0.0
        std_t = statistics.stdev(tests) if len(tests) > 1 else 0.0
        print(f"  → {model_name:30s} "
              f"val_R2={statistics.mean(vals):.4f}±{std_v:.4f}  "
              f"test_R2={statistics.mean(tests):.4f}±{std_t:.4f}") # ghgjnb


  Tuning on: q_OnlineNewsPopularity

  [HPO] MLP — 50 trials...


  0%|          | 0/50 [00:00<?, ?it/s]

  [HPO] best_val_R2=0.1514  hp={'n_layers': 4, 'hidden_dim': 128, 'dropout': 0.3}  lr=8.88e-04
  [MLP]  seed=42  (tuned)
  [   1] train_loss=0.9730  val_loss=0.8965  val_R2=0.0795  *
  [  10] train_loss=0.8316  val_loss=0.8351  val_R2=0.1425
  [  20] train_loss=0.7884  val_loss=0.8355  val_R2=0.1421
  [  30] train_loss=0.7458  val_loss=0.8413  val_R2=0.1362
  Early stop at epoch 30 (patience=16)
  → best_epoch=14  best_val_R2=0.1514  test_R2=0.1404  (8.2s)
  [MLP]  seed=43  (tuned)
  [   1] train_loss=0.9755  val_loss=0.9034  val_R2=0.0724  *
  [  10] train_loss=0.8341  val_loss=0.8348  val_R2=0.1429
  [  20] train_loss=0.7832  val_loss=0.8386  val_R2=0.1390
  [  30] train_loss=0.7490  val_loss=0.8452  val_R2=0.1322
  Early stop at epoch 33 (patience=16)
  → best_epoch=17  best_val_R2=0.1460  test_R2=0.1354  (8.9s)
  [MLP]  seed=44  (tuned)
  [   1] train_loss=0.9744  val_loss=0.9038  val_R2=0.0720  *
  [  10] train_loss=0.8297  val_loss=0.8413  val_R2=0.1362
  [  20] train_loss=0.7835

  0%|          | 0/50 [00:00<?, ?it/s]

  [HPO] best_val_R2=0.1576  hp={'n_layers': 3, 'hidden_dim': 256, 'dropout': 0.1, 'k': 8, 'sigma': 0.08325295943065981}  lr=1.15e-04
  [EmbMLP-orig-plain]  seed=42  (tuned)
  [   1] train_loss=0.9569  val_loss=0.8913  val_R2=0.0849  *
  [  10] train_loss=0.8176  val_loss=0.8358  val_R2=0.1419
  [  20] train_loss=0.7805  val_loss=0.8305  val_R2=0.1472
  [  30] train_loss=0.7298  val_loss=0.8438  val_R2=0.1336
  Early stop at epoch 30 (patience=16)
  → best_epoch=14  best_val_R2=0.1576  test_R2=0.1488  (8.7s)
  [EmbMLP-orig-plain]  seed=43  (tuned)
  [   1] train_loss=0.9552  val_loss=0.8862  val_R2=0.0901  *
  [  10] train_loss=0.8157  val_loss=0.8346  val_R2=0.1430
  [  20] train_loss=0.7705  val_loss=0.8333  val_R2=0.1444
  [  30] train_loss=0.7120  val_loss=0.8525  val_R2=0.1247
  Early stop at epoch 33 (patience=16)
  → best_epoch=17  best_val_R2=0.1468  test_R2=0.1429  (9.5s)
  [EmbMLP-orig-plain]  seed=44  (tuned)
  [   1] train_loss=0.9611  val_loss=0.8919  val_R2=0.0843  *
  [  

...

  0%|          | 0/50 [00:00<?, ?it/s]

  [HPO] best_val_R2=0.1493  hp={'n_layers': 2, 'hidden_dim': 512, 'dropout': 0.5, 'k': 48, 'sigma': 0.0011135175870974378}  lr=2.10e-04
  [EmbMLP-grid-sc]  seed=42  (tuned)
  [   1] train_loss=0.9522  val_loss=0.8612  val_R2=0.1157  *
  [  10] train_loss=0.7631  val_loss=0.8427  val_R2=0.1348
  [  20] train_loss=0.6351  val_loss=0.8686  val_R2=0.1082
  Early stop at epoch 20 (patience=16)
  → best_epoch=4  best_val_R2=0.1493  test_R2=0.1432  (6.5s)
  [EmbMLP-grid-sc]  seed=43  (tuned)
  [   1] train_loss=0.9544  val_loss=0.8727  val_R2=0.1040  *
  [  10] train_loss=0.7574  val_loss=0.8444  val_R2=0.1330
  [  20] train_loss=0.6294  val_loss=0.8790  val_R2=0.0975
  Early stop at epoch 20 (patience=16)
  → best_epoch=4  best_val_R2=0.1442  test_R2=0.1380  (6.5s)
  [EmbMLP-grid-sc]  seed=44  (tuned)
  [   1] train_loss=0.9543  val_loss=0.8596  val_R2=0.1174  *
  [  10] train_loss=0.7631  val_loss=0.8335  val_R2=0.1442
  [  20] train_loss=0.6662  val_loss=0.8637  val_R2=0.1132
  Early stop 

  0%|          | 0/50 [00:00<?, ?it/s]

  [HPO] best_val_R2=0.1498  hp={'n_layers': 4, 'hidden_dim': 256, 'dropout': 0.3, 'k': 96, 'sigma': 0.07105724685332379}  lr=8.06e-04
  [EmbMLP-grid-sc_af]  seed=42  (tuned)
  [   1] train_loss=0.9699  val_loss=0.8964  val_R2=0.0796  *
  [  10] train_loss=0.7580  val_loss=0.8817  val_R2=0.0947
  [  20] train_loss=0.6121  val_loss=0.8914  val_R2=0.0847
  Early stop at epoch 20 (patience=16)
  → best_epoch=4  best_val_R2=0.1498  test_R2=0.1424  (7.2s)
  [EmbMLP-grid-sc_af]  seed=43  (tuned)
  [   1] train_loss=0.9744  val_loss=0.8565  val_R2=0.1206  *
  [  10] train_loss=0.7194  val_loss=0.8711  val_R2=0.1056
  Early stop at epoch 19 (patience=16)
  → best_epoch=3  best_val_R2=0.1400  test_R2=0.1368  (6.9s)
  [EmbMLP-grid-sc_af]  seed=44  (tuned)
  [   1] train_loss=0.9774  val_loss=0.8631  val_R2=0.1138  *
  [  10] train_loss=0.7481  val_loss=0.9157  val_R2=0.0599
  [  20] train_loss=0.5955  val_loss=0.8892  val_R2=0.0870
  Early stop at epoch 21 (patience=16)
  → best_epoch=5  best_val

  0%|          | 0/50 [00:00<?, ?it/s]

## Results: Default HPs vs Tuned HPs

Side-by-side comparison, test R² (mean ± std over seeds).

In [ ]:
import pandas as pd # kbbbb

def _agg(results, ds_name):
    """Return {model_name: (test_mean, test_std)} for one dataset."""
    from collections import defaultdict
    groups = defaultdict(list)
    for r in results:
        if r['dataset_name'] == ds_name:
            groups[r['model_name']].append(r['test_metric'])
    out = {}
    for model, vals in groups.items():
        out[model] = (
            statistics.mean(vals),
            statistics.stdev(vals) if len(vals) > 1 else 0.0,
        )
    return out


for ds_name in DATASETS:
    agg_def   = _agg(results_default, ds_name)
    agg_tuned = _agg(results_tuned,   ds_name)

    rows = []
    baseline_def   = agg_def.get('MLP',   (None, None))[0]
    baseline_tuned = agg_tuned.get('MLP', (None, None))[0]

    for model_name in ALL_VARIANTS:
        d = agg_def.get(model_name)
        t = agg_tuned.get(model_name)

        def _fmt(tup):
            if tup is None: return '—'
            return f'{tup[0]:.4f} ±{tup[1]:.4f}'

        def _delta(tup, baseline):
            if tup is None or baseline is None: return ''
            if model_name == 'MLP': return ''
            delta = tup[0] - baseline
            arrow = '▲' if delta > 0 else '▼'
            return f'{arrow}{abs(delta):.4f}'

        rows.append({
            'Model':          model_name,
            'Default Test R²': _fmt(d),
            'Δ (vs MLP)':     _delta(d, baseline_def),
            'Tuned Test R²':  _fmt(t),
            'Δ (vs MLP)  ':   _delta(t, baseline_tuned),
        })

    df = pd.DataFrame(rows).set_index('Model')
    print(f'\nDataset: {ds_name}  (Test R² ↑, mean ± std over {N_SEEDS} seeds)\n')
    display(df)

## Save tuning info

In [ ]:
import os
os.makedirs(RESULTS_DIR, exist_ok=True)

# Save tuning results (without optuna Study objects)
tuning_slim = {
    ds: {
        model: {'best_hp': t['best_hp'], 'best_lr': t['best_lr'],
                'best_wd': t['best_wd'], 'best_val_r2': t['best_val_r2']}
        for model, t in models.items()
    }
    for ds, models in tuning_info.items()
}
with open(RESULTS_DIR + '/tuning_info.json', 'w') as f:
    json.dump(tuning_slim, f, indent=2)

# Save evaluation results
all_results = results_default + results_tuned
with open(RESULTS_DIR + '/all_results.json', 'w') as f:
    json.dump(
        [{k: v for k, v in r.items() if k != 'history'} for r in all_results],
        f, indent=2
    )

print('Saved to', RESULTS_DIR)

## Learning curves

In [ ]:
import matplotlib.pyplot as plt

PLOT_VARIANTS = ['MLP', 'EmbMLP-grid-sc_af', 'PAFNet-sc_af-ln']
DATASET       = DATASETS[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (results, title) in zip(axes, [
    (results_default, 'Default HPs'),
    (results_tuned,   'Tuned HPs'),
]):
    for r in results:
        if (r['dataset_name'] == DATASET
                and r['model_name'] in PLOT_VARIANTS
                and r['seed'] == 42
                and 'history' in r):
            epochs = [h['epoch']      for h in r['history']]
            vals   = [h['val_metric'] for h in r['history']]
            ax.plot(epochs, vals, label=r['model_name'])
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val R²')
    ax.set_title(f'{DATASET} — {title}')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR + '/learning_curves.png', dpi=150)
plt.show()